<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-10-tuning-and-evaluation/lesson-10.1-sft-lora/notebooks/GCP_Capstone_10.1_SFT_LoRA.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10.1 SFT with LoRA — Fine-Tune Gemini Flash on Vertex AI
**Netsetos GenAI Engineering — GCP Capstone**


In [ ]:
!pip install -q google-genai google-cloud-aiplatform pandas
from google import genai
from google.genai import types
import json, time, os

PROJECT = 'your-project-id'
LOCATION = 'us-central1'
BUCKET = 'your-bucket'  # Cloud Storage bucket

client = genai.Client(enterprise=True, project=PROJECT, location=LOCATION)
print('SDK ready')


## Cell 1: Generate JSONL Training Data


In [ ]:
# DocuMind document classification training data
training_examples = [
    {
        'doc': 'Invoice #INV-2024-0892\nBill To: Acme Corp\nDate: March 15, 2024\nItem: Cloud Services - Q1\nAmount: $45,000.00\nPayment: Net 30',
        'label': 'INVOICE'
    },
    {
        'doc': 'Service Agreement\nBetween: TechCorp and ClientX\nEffective: January 1, 2024\nTerm: 12 months\nThis agreement outlines the terms and conditions of service delivery...',
        'label': 'CONTRACT'
    },
    {
        'doc': 'Q1 2024 Financial Report\nExecutive Summary: Revenue increased 23% YoY...\nOperating margin improved to 18%.\nKey insights and forward-looking statements follow.',
        'label': 'REPORT'
    },
    {
        'doc': 'Receipt #R-2024-5521\nDate: 2024-03-20\nStore: OfficeSupplies Inc.\nItems: Printer paper - $25.50, Ink cartridges - $89.00\nTotal: $114.50\nPayment: Credit Card',
        'label': 'RECEIPT'
    },
    {
        'doc': 'From: john@client.com\nTo: support@documind.ai\nSubject: Question about API rate limits\n\nHi team, I am hitting rate limits on my production deployment...',
        'label': 'CORRESPONDENCE'
    },
]

# Build JSONL with proper Gemini format
system_prompt = 'You are DocuMind, a document classifier. Classify documents as: INVOICE, CONTRACT, REPORT, RECEIPT, or CORRESPONDENCE. Respond with the category name only.'

jsonl_lines = []
for ex in training_examples:
    record = {
        'systemInstruction': {'role': 'system', 'parts': [{'text': system_prompt}]},
        'contents': [
            {'role': 'user', 'parts': [{'text': f'Classify:\n\n{ex["doc"]}'}]},
            {'role': 'model', 'parts': [{'text': ex['label']}]}
        ]
    }
    jsonl_lines.append(json.dumps(record))

with open('train.jsonl', 'w') as f:
    f.write('\n'.join(jsonl_lines))

print(f'Created train.jsonl with {len(jsonl_lines)} examples')
print('\nFirst example:')
print(jsonl_lines[0][:200] + '...')


## Cell 2: Validate JSONL Format


In [ ]:
# Validation: catch common JSONL mistakes before training
def validate_jsonl(path):
    errors = []
    with open(path) as f:
        for i, line in enumerate(f, 1):
            try:
                rec = json.loads(line)
            except json.JSONDecodeError as e:
                errors.append(f'Line {i}: Invalid JSON - {e}')
                continue
            
            # Check contents exists
            if 'contents' not in rec:
                errors.append(f'Line {i}: Missing contents field')
                continue
            
            contents = rec['contents']
            roles = [c.get('role') for c in contents]
            
            # Check last entry is model
            if roles[-1] != 'model':
                errors.append(f'Line {i}: Last entry must be role=model, got {roles[-1]}')
            
            # Check no assistant role
            if 'assistant' in roles:
                errors.append(f'Line {i}: Use role=model, not role=assistant')
            
            # Check roles alternate
            for j, r in enumerate(roles):
                expected = 'user' if j % 2 == 0 else 'model'
                if r != expected:
                    errors.append(f'Line {i}: Position {j} should be {expected}, got {r}')
                    break
            
            # Check parts use text field (common mistake: content)
            for j, c in enumerate(contents):
                for p in c.get('parts', []):
                    if 'content' in p and 'text' not in p:
                        errors.append(f'Line {i}: Use "text" not "content" in parts')
    
    return errors

errors = validate_jsonl('train.jsonl')
if errors:
    for e in errors[:5]:
        print('ERROR:', e)
else:
    print('OK: JSONL format is valid')


## Cell 3: Function Calling JSONL Example


In [ ]:
# Function calling training example (pattern: model generates function call)
fc_example = {
    'system_instruction': {
        'role': 'system',
        'parts': [{'text': 'You are DocuMind. Use the provided tools to process documents.'}]
    },
    'contents': [
        {'role': 'user', 'parts': [{'text': 'I uploaded a new document. Please classify it.'}]},
        {'role': 'model', 'parts': [{
            'functionCall': {
                'name': 'classify_document',
                'args': {'document_id': 'current_doc', 'confidence_threshold': 0.85}
            }
        }]}
    ],
    'tools': [{
        'functionDeclarations': [
            {
                'name': 'classify_document',
                'description': 'Classify a document into a category',
                'parameters': {
                    'type': 'OBJECT',
                    'properties': {
                        'document_id': {'type': 'STRING'},
                        'confidence_threshold': {'type': 'NUMBER'}
                    },
                    'required': ['document_id']
                }
            }
        ]
    }]
}
print(json.dumps(fc_example, indent=2)[:500] + '...')


## Cell 4: Cost Estimation


In [ ]:
# Estimate training cost before launching
import json

def estimate_training_cost(jsonl_path, epochs=3, per_token_rate=0.0000035):
    '''Rough estimate: 1 token ~ 4 characters for English.'''
    total_chars = 0
    n_examples = 0
    with open(jsonl_path) as f:
        for line in f:
            rec = json.loads(line)
            total_chars += len(json.dumps(rec))
            n_examples += 1
    
    approx_tokens = total_chars // 4
    training_tokens = approx_tokens * epochs
    cost = training_tokens * per_token_rate
    
    print(f'Examples: {n_examples}')
    print(f'Approx tokens per example: {approx_tokens // n_examples if n_examples else 0}')
    print(f'Total approx tokens: {approx_tokens:,}')
    print(f'Training tokens (x {epochs} epochs): {training_tokens:,}')
    print(f'Estimated training cost: ${cost:.4f}')
    print(f'(Inference cost: SAME as base model - LoRA adapters are free to serve)')

estimate_training_cost('train.jsonl', epochs=3)


## Cell 5: Launch Fine-Tuning Job (Template)


In [ ]:
# Template: actual launch requires valid GCS bucket and project
# Upload training JSONL to GCS first:
# !gsutil cp train.jsonl gs://{BUCKET}/train.jsonl

def launch_tuning_job(base_model='gemini-3.6-flash', 
                      train_uri='gs://bucket/train.jsonl',
                      val_uri=None, epochs=3, 
                      display_name='documind-classifier-v1'):
    '''Launch a supervised fine-tuning job.'''
    config_kwargs = {
        'epoch_count': epochs,
        'tuned_model_display_name': display_name,
    }
    
    dataset_kwargs = {'gcs_uri': train_uri}
    
    tuning_job = client.tunings.tune(
        base_model=base_model,
        training_dataset=types.TuningDataset(**dataset_kwargs),
        config=types.CreateTuningJobConfig(**config_kwargs)
    )
    return tuning_job

# Poll pattern
def poll_until_complete(tuning_job):
    completed = {'JOB_STATE_SUCCEEDED', 'JOB_STATE_FAILED', 'JOB_STATE_CANCELLED'}
    while str(tuning_job.state) not in completed:
        print(f'Status: {tuning_job.state}')
        tuning_job = client.tunings.get(name=tuning_job.name)
        time.sleep(30)
    return tuning_job

print('Functions defined. Uncomment to launch:')
print('# job = launch_tuning_job(train_uri=f"gs://{BUCKET}/train.jsonl")')
print('# job = poll_until_complete(job)')
print('# print(f"Tuned endpoint: {job.tuned_model.endpoint}")')


## Cell 6: A/B Evaluation Pattern


In [ ]:
# Pointwise + pairwise evaluation template
import pandas as pd

def build_eval_dataset():
    '''Build evaluation dataset from holdout examples.'''
    return pd.DataFrame({
        'prompt': [
            'Classify: Invoice #2024-001, Amount: $4,590, Due: 2024-04-14',
            'Classify: Service Agreement between A and B, Term: 24 months',
            'Classify: Q3 Financial Report showing 18% YoY growth',
        ],
        'reference': ['INVOICE', 'CONTRACT', 'REPORT'],
    })

eval_df = build_eval_dataset()
print('Evaluation dataset:')
print(eval_df)

print('\n--- A/B Evaluation Pattern ---')
print('''
from vertexai.evaluation import EvalTask, PairwiseMetric, MetricPromptTemplateExamples

# Pointwise
base_result = EvalTask(
    dataset=eval_df, metrics=["exact_match", "rouge_l_sum"],
    experiment="ab-test"
).evaluate(model="gemini-3.6-flash",
          experiment_run_name="base")

tuned_result = EvalTask(
    dataset=eval_df, metrics=["exact_match", "rouge_l_sum"],
    experiment="ab-test"
).evaluate(model=tuned_endpoint,
          experiment_run_name="tuned")

# Compare
print(f"Base exact_match: {base_result.summary_metrics[\"exact_match/mean\"]:.2%}")
print(f"Tuned exact_match: {tuned_result.summary_metrics[\"exact_match/mean\"]:.2%}")
''')


## Cell 7: Decision Framework - Fine-Tune or Prompt?


In [ ]:
# Decision helper: should you fine-tune?
def should_finetune(
    num_examples, 
    monthly_calls, 
    format_consistency_needed,
    domain_specific_terms,
    requirements_stable
):
    '''Returns recommendation + reasoning.'''
    score = 0
    reasons = []
    
    if num_examples >= 100:
        score += 2
        reasons.append(f'+ {num_examples} examples (>=100 threshold)')
    else:
        reasons.append(f'- Only {num_examples} examples (<100). Use few-shot instead')
    
    if monthly_calls > 50000:
        score += 2
        reasons.append(f'+ High volume ({monthly_calls:,}/month). Few-shot overhead expensive')
    
    if format_consistency_needed:
        score += 2
        reasons.append('+ Format consistency needed (JSON schemas, labels)')
    
    if domain_specific_terms:
        score += 1
        reasons.append('+ Domain-specific terminology')
    
    if not requirements_stable:
        score -= 3
        reasons.append('- Requirements change frequently. Prompts are faster to update')
    
    recommendation = 'FINE-TUNE' if score >= 3 else 'USE PROMPTING'
    return recommendation, reasons, score

# Test: DocuMind classifier
rec, reasons, score = should_finetune(
    num_examples=200,
    monthly_calls=100000,
    format_consistency_needed=True,
    domain_specific_terms=True,
    requirements_stable=True
)
print(f'Recommendation: {rec} (score: {score})')
for r in reasons:
    print(f'  {r}')


## Done!
- JSONL dataset preparation with systemInstruction, contents, roles
- Validation of common JSONL mistakes
- Function calling JSONL with tools/functionCall/functionResponse
- Training cost estimation
- Tuning job launch pattern with google-genai SDK
- A/B evaluation with pointwise + pairwise metrics
- Decision framework for fine-tune vs prompt
